In [1]:

from google.colab import files

uploaded = files.upload()


Saving archive (2).zip to archive (2).zip


In [3]:

import pandas as pd
import os
import zipfile

print("Uploaded files:")

for filename in uploaded.keys():
    print(filename)


Uploaded files:
archive (2).zip


In [4]:

import os
import zipfile

for filename in uploaded.keys():
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as z:
            z.extractall("/content/customer_data")

for root, dirs, filenames in os.walk("/content/customer_data"):
    for filename in filenames:
        if filename.endswith(".csv"):
            print(os.path.join(root, filename))


/content/customer_data/Bitext_Sample_Customer_Service_Testing_Dataset.csv
/content/customer_data/Bitext_Sample_Customer_Service_Validation_Dataset.csv
/content/customer_data/Bitext_Sample_Customer_Service_Training_Dataset.csv


In [5]:

import glob
import pandas as pd
import os
csv_files = glob.glob(
    "/content/customer_data/**/*.csv",
    recursive=True
)

tables = {}

for i, file in enumerate(csv_files, start=1):
    tables[f"table{i}"] = pd.read_csv(file)

    print(f"\n--- TABLE {i}: {os.path.basename(file)} ---")
    print("Shape:", tables[f"table{i}"].shape)
    print("Columns:", tables[f"table{i}"].columns.tolist())
    display(tables[f"table{i}"].head())



--- TABLE 1: Bitext_Sample_Customer_Service_Testing_Dataset.csv ---
Shape: (818, 4)
Columns: ['utterance', 'intent', 'category', 'tags']


,utterance,intent,category,tags
0,I have a question about cancelling an order,cancel_order,ORDER,B
1,help canceling the order I have made,cancel_order,ORDER,B
2,I do not know how to cancel an order I have made,cancel_order,ORDER,BE
3,where can I cancel the last order I made?,cancel_order,ORDER,BI
4,I don't want the last order I made,cancel_order,ORDER,B



--- TABLE 2: Bitext_Sample_Customer_Service_Validation_Dataset.csv ---
Shape: (818, 4)
Columns: ['utterance', 'intent', 'category', 'tags']


,utterance,intent,category,tags
0,I want help to cancel the order I have made,cancel_order,ORDER,B
1,where can I get information about order cancel...,cancel_order,ORDER,BIM
2,help to cancel the last order,cancel_order,ORDER,B
3,I need help with canceling an order I made,cancel_order,ORDER,B
4,I do not know how I can cancel the order I made,cancel_order,ORDER,BE



--- TABLE 3: Bitext_Sample_Customer_Service_Training_Dataset.csv ---
Shape: (6539, 4)
Columns: ['utterance', 'intent', 'category', 'tags']


,utterance,intent,category,tags
0,would it be possible to cancel the order I made?,cancel_order,ORDER,BIP
1,cancelling order,cancel_order,ORDER,BK
2,I need assistance canceling the last order I h...,cancel_order,ORDER,B
3,problem with canceling the order I made,cancel_order,ORDER,B
4,I don't know how to cancel the order I made,cancel_order,ORDER,B


In [6]:

# Display column names for all three tables

for name, table in tables.items():
    print(f"\n{name}")
    print("Columns:", table.columns.tolist())
    print("Shape:", table.shape)
    display(table.head(3))



table1
Columns: ['utterance', 'intent', 'category', 'tags']
Shape: (818, 4)


,utterance,intent,category,tags
0,I have a question about cancelling an order,cancel_order,ORDER,B
1,help canceling the order I have made,cancel_order,ORDER,B
2,I do not know how to cancel an order I have made,cancel_order,ORDER,BE



table2
Columns: ['utterance', 'intent', 'category', 'tags']
Shape: (818, 4)


,utterance,intent,category,tags
0,I want help to cancel the order I have made,cancel_order,ORDER,B
1,where can I get information about order cancel...,cancel_order,ORDER,BIM
2,help to cancel the last order,cancel_order,ORDER,B



table3
Columns: ['utterance', 'intent', 'category', 'tags']
Shape: (6539, 4)


,utterance,intent,category,tags
0,would it be possible to cancel the order I made?,cancel_order,ORDER,BIP
1,cancelling order,cancel_order,ORDER,BK
2,I need assistance canceling the last order I h...,cancel_order,ORDER,B


In [7]:

import pandas as pd

# Combine the three tables row by row
df = pd.concat(
    [tables["table1"], tables["table2"], tables["table3"]],
    ignore_index=True
)

print("Merged dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nNumber of duplicate rows:", df.duplicated().sum())


Merged dataset shape: (8175, 4)

First five rows:


,utterance,intent,category,tags
0,I have a question about cancelling an order,cancel_order,ORDER,B
1,help canceling the order I have made,cancel_order,ORDER,B
2,I do not know how to cancel an order I have made,cancel_order,ORDER,BE
3,where can I cancel the last order I made?,cancel_order,ORDER,BI
4,I don't want the last order I made,cancel_order,ORDER,B



Number of duplicate rows: 0


In [8]:

print("Number of unique intents:", df["intent"].nunique())

print("\nExamples per intent:")
display(df["intent"].value_counts().head(10))

print("\nCategory distribution:")
display(df["category"].value_counts())


Number of unique intents: 27

Examples per intent:


,count
intent,
check_invoice,324
get_invoice,324
payment_issue,323
review,315
track_refund,308
set_up_shipping_address,307
place_order,306
cancel_order,305
track_order,305



Category distribution:


,count
category,
ACCOUNT,1774
ORDER,1220
REFUND,904
INVOICE,648
PAYMENT,620
FEEDBACK,613
SHIPPING_ADDRESS,604
DELIVERY,603
CONTACT,596


In [9]:

# Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Check exact duplicate rows
print("\nExact duplicate rows:", df.duplicated().sum())

# Check duplicate customer messages
print("Duplicate utterances:", df["utterance"].duplicated().sum())

# Check whether any intent labels are missing
print("Missing intent labels:", df["intent"].isnull().sum())


Missing values in each column:
utterance    0
intent       0
category     0
tags         0
dtype: int64

Exact duplicate rows: 0
Duplicate utterances: 0
Missing intent labels: 0


In [10]:

from sklearn.model_selection import train_test_split

# Input messages
X = df["utterance"]

# Target labels
y = df["intent"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Total samples:", len(df))
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining intent distribution:")
print(y_train.value_counts().head())


Total samples: 8175
Training samples: 6540
Testing samples: 1635

Training intent distribution:
intent
check_invoice    259
get_invoice      259
payment_issue    258
review           252
track_refund     247
Name: count, dtype: int64


In [11]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the text-to-numbers converter
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    stop_words="english"
)

# Learn vocabulary from training messages only
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test messages using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training feature matrix:", X_train_tfidf.shape)
print("Testing feature matrix:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


Training feature matrix: (6540, 2771)
Testing feature matrix: (1635, 2771)
Vocabulary size: 2771


In [12]:

from sklearn.linear_model import LogisticRegression

# Create the ML model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
model.fit(X_train_tfidf, y_train)

print("Model training completed successfully!")


Model training completed successfully!


In [13]:

from sklearn.metrics import accuracy_score, classification_report

# Predict intents for test messages
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Accuracy Percentage: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Test Accuracy: 0.9761
Test Accuracy Percentage: 97.61%

Classification Report:
                          precision    recall  f1-score   support

            cancel_order       1.00      0.98      0.99        61
            change_order       1.00      1.00      1.00        61
 change_shipping_address       1.00      1.00      1.00        59
  check_cancellation_fee       1.00      1.00      1.00        60
           check_invoice       0.93      0.62      0.74        65
   check_payment_methods       1.00      1.00      1.00        59
     check_refund_policy       1.00      1.00      1.00        60
               complaint       0.98      1.00      0.99        60
contact_customer_service       1.00      1.00      1.00        60
     contact_human_agent       0.98      1.00      0.99        59
          create_account       0.97      0.93      0.95        60
          delete_account       0.97      0.98      0.98        60
        delivery_options       1.00      1.00      1.00       

In [14]:

message = ["I need help cancelling my recent order"]

# Convert the message using the existing TF-IDF vocabulary
message_tfidf = tfidf.transform(message)

# Predict the intent
predicted_intent = model.predict(message_tfidf)[0]

print("Customer message:", message[0])
print("Predicted intent:", predicted_intent)


Customer message: I need help cancelling my recent order
Predicted intent: cancel_order


In [17]:

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# 1. Parameters to test
param_grid = {
    "C": [0.1, 1, 10]
}

# 2. Create the model
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# 3. Search for the best parameter
grid_search = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# 4. Train and compare parameter combinations
grid_search.fit(X_train_tfidf, y_train)

# 5. Display the results
print("Best Parameters:", grid_search.best_params_)
print(
    "Best Cross-Validation Accuracy:",
    round(grid_search.best_score_ * 100, 2),
    "%"
)


Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best Parameters: {'C': 10}
Best Cross-Validation Accuracy: 97.89 %


In [18]:

from sklearn.metrics import accuracy_score, classification_report

# Get the best model found during tuning
tuned_model = grid_search.best_estimator_

# Predict intents for unseen test messages
y_pred_tuned = tuned_model.predict(X_test_tfidf)

# Calculate test accuracy
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print("Baseline Test Accuracy: 97.49%")
print(f"Tuned Test Accuracy: {tuned_accuracy * 100:.2f}%")

print("\nTuned Model Classification Report:")
print(classification_report(
    y_test,
    y_pred_tuned,
    zero_division=0
))


Baseline Test Accuracy: 97.49%
Tuned Test Accuracy: 97.92%

Tuned Model Classification Report:
                          precision    recall  f1-score   support

            cancel_order       1.00      0.98      0.99        61
            change_order       1.00      1.00      1.00        61
 change_shipping_address       1.00      1.00      1.00        59
  check_cancellation_fee       1.00      1.00      1.00        60
           check_invoice       0.89      0.65      0.75        65
   check_payment_methods       1.00      1.00      1.00        59
     check_refund_policy       1.00      1.00      1.00        60
               complaint       1.00      1.00      1.00        60
contact_customer_service       1.00      1.00      1.00        60
     contact_human_agent       1.00      1.00      1.00        59
          create_account       1.00      0.97      0.98        60
          delete_account       0.98      1.00      0.99        60
        delivery_options       1.00      1.00 